пункт a

H0: возраст не влияет на средние показатели

H1: хотя бы 1 среднее отличается

In [11]:
import numpy as np
from scipy import stats

In [12]:
alpha = 0.05

g1 = [83, 85]
g2 = [84, 85, 85, 86, 86, 87]
g3 = [86, 87, 87, 87, 88, 88, 88, 88, 88, 89, 90]
g4 = [89, 90, 90, 91]
g5 = [90, 92]

In [13]:
Y = np.array(
    g1 + g2 + g3 + g4 + g5
)
groups = (
    [1]*len(g1) +
    [2]*len(g2) +
    [3]*len(g3) +
    [4]*len(g4) +
    [5]*len(g5)
)
groups = np.array(groups)
n = len(Y)

In [14]:
z1 = (groups == 1).astype(int)
z2 = (groups == 2).astype(int)
z3 = (groups == 3).astype(int)
z4 = (groups == 4).astype(int)
z5 = (groups == 5).astype(int)

X = np.column_stack((z1, z2, z3, z4, z5))

In [16]:
p = X.shape[1]

F = X.T @ X
F_inv = np.linalg.inv(F)

beta = F_inv @ X.T @ Y

Y_pred = X @ beta

E = Y - Y_pred

RSS = E.T @ E

TSS = (Y - np.mean(Y)).T @ (Y - np.mean(Y))

R2 = 1 - RSS / TSS

delta = R2 * (n - p) / ((1 - R2) * (p - 1))

p_value = 1 - stats.f.cdf(
    delta,
    dfn=p - 1,
    dfd=n - p
)

print('beta =', beta)
print('RSS =', RSS)
print('TSS =', TSS)
print('R^2 =', R2)
print('delta =', delta)
print('p-value =', p_value)

if p_value < alpha:
    print('Отвергаем H0: возраст влияет на содержание IgA')
else:
    print('Нет оснований отвергнуть H0: влияние возраста статистически не подтверждено')

beta = [84.         85.5        87.81818182 90.         91.        ]
RSS = 23.136363636363637
TSS = 122.16000000000003
R^2 = 0.8106060606060607
delta = 21.40000000000001
p-value = 5.407435041959729e-07
Отвергаем H0: возраст влияет на содержание IgA


пункт b

In [17]:
pairs = []

for i in range(p):
    for j in range(i + 1, p):

        se_ij = np.sqrt(
            RSS / (n - p) * (F_inv[i, i] + F_inv[j, j])
        )

        delta_ij = abs(beta[i] - beta[j]) / se_ij

        p_value_ij = 2 * (1 - stats.t.cdf(delta_ij, df=n - p))

        pairs.append([
            i + 1,
            j + 1,
            beta[i],
            beta[j],
            delta_ij,
            p_value_ij
        ])

In [18]:
pairs_sorted = sorted(pairs, key=lambda x: x[5])

m = len(pairs_sorted)

print('Попарные сравнения с поправкой Холма-Бонферрони:')
print()

stop = False

for k, row in enumerate(pairs_sorted):
    i, j, beta_i, beta_j, delta_ij, p_value_ij = row

    alpha_k = alpha / (m - k)

    print(f'Группы {i} и {j}')
    print(f'beta_{i} = {beta_i:.3f}, beta_{j} = {beta_j:.3f}')
    print(f'delta = {delta_ij:.3f}')
    print(f'p-value = {p_value_ij:.5f}')
    print(f'alpha_k = {alpha_k:.5f}')

    if (not stop) and p_value_ij < alpha_k:
        print('Отвергаем H0: средние различаются')
    else:
        stop = True
        print('Нет оснований отвергнуть H0: различие не подтверждено')


Попарные сравнения с поправкой Холма-Бонферрони:

Группы 1 и 5
beta_1 = 84.000, beta_5 = 91.000
delta = 6.508
p-value = 0.00000
alpha_k = 0.00500
Отвергаем H0: средние различаются
Группы 2 и 4
beta_2 = 85.500, beta_4 = 90.000
delta = 6.482
p-value = 0.00000
alpha_k = 0.00556
Отвергаем H0: средние различаются
Группы 1 и 4
beta_1 = 84.000, beta_4 = 90.000
delta = 6.442
p-value = 0.00000
alpha_k = 0.00625
Отвергаем H0: средние различаются
Группы 2 и 5
beta_2 = 85.500, beta_5 = 91.000
delta = 6.263
p-value = 0.00000
alpha_k = 0.00714
Отвергаем H0: средние различаются
Группы 1 и 3
beta_1 = 84.000, beta_3 = 87.818
delta = 4.618
p-value = 0.00017
alpha_k = 0.00833
Отвергаем H0: средние различаются
Группы 2 и 3
beta_2 = 85.500, beta_3 = 87.818
delta = 4.247
p-value = 0.00040
alpha_k = 0.01000
Отвергаем H0: средние различаются
Группы 3 и 5
beta_3 = 87.818, beta_5 = 91.000
delta = 3.848
p-value = 0.00100
alpha_k = 0.01250
Отвергаем H0: средние различаются
Группы 3 и 4
beta_3 = 87.818, beta_4 = 9